# Continual Learning in MemoRizz: Workflows to Skills

## Part I - Why Workflows Become Skills

This guide follows one continuous chain: define the learning contract, establish a live
runtime, capture business-graded experience, distill and govern a skill, measure its
effect, and clean up the run. Each part ends with the claims it established and the
question carried into the next part.

A workflow is evidence: one concrete sequence of tool calls, arguments, results, and
errors. A skill is a reviewed, reusable instruction distilled from several comparable
workflows. Moving from workflows to skills is useful because it changes successful
behavior from passive history into an instruction artifact that can be selected before
the next action.

There is an important precision point behind the phrase **instruction privilege**.
Models do not grant every file named `SKILL.md` a universal higher rank. They follow the
message and instruction hierarchy implemented by the host system. OpenAI's
[instruction-hierarchy research](https://arxiv.org/abs/2404.13208) describes system and
developer instructions as more privileged than user or tool text. OpenAI
[Codex Skills](https://learn.chatgpt.com/docs/customization/overview#skills) package
reusable workflows as instructions that are loaded when relevant. Anthropic documents
that [skill metadata is placed in Claude's system prompt and full instructions are
loaded on demand](https://platform.claude.com/docs/en/build-with-claude/skills-guide).
Google similarly says
[system instructions are processed before prompts and apply across turns](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/prompts/system-instruction-introduction).

The defensible conclusion is conditional: **a validated skill can receive higher
effective priority when the runtime loads it into a privileged or specially managed
instruction channel**. Raw workflow records do not get that benefit by themselves.
MemoRizz currently injects matching learned-skill content into the per-turn context above
ordinary recalled memories; it does not mislabel that content as a system message.
Accordingly, learned skills remain **strong priors, not mandates**: preconditions must be
checked, deviations are allowed, and later outcomes can demote a bad skill.

```mermaid
flowchart TD
    Q[User request] --> A[MemAgent with live Toolbox]
    A --> T[Selected LLM calls real Python tools]
    T --> W[(Oracle workflow memory)]
    W --> C[Canonicalize trajectory]
    C --> G{Promotion gates pass?}
    G -- No --> W
    G -- Yes --> D[Selected LLM distills and validates]
    D --> H[Shadow skill review]
    H -- Approved --> S[(Active Oracle Skillbox skill)]
    H -- Rejected --> W
    S --> E[Oracle in-database vector retrieval]
    E --> P[Inject matching skill as a strong prior]
    P --> A
    A --> M[Grade outcomes and monitor drift]
    M -- Healthy --> S
    M -- Regressed or stale --> X[Demote skill]
    X --> W
```

### Glossary

| Term | Definition in this guide |
|---|---|
| **Continual learning** | An operational loop that keeps collecting outcomes after deployment and updates reusable behavior from new evidence. Here it updates workflow and skill memory, not model weights. |
| **Self-improving agent** | An agent system whose measured behavior can improve through its own captured experience, validation, retrieval, and rollback loop. This does not imply autonomous retraining or unconstrained self-modification. |
| **Workflow / trajectory** | One observed run: the ordered tool calls, argument values, results, errors, query, and outcome. |
| **Canonical signature** | A value-independent representation of a trajectory: ordered tool names, argument-key shapes, and error flags, with immediate retries collapsed. |
| **Canonical hash / trajectory class** | A stable hash of the canonical signature. Runs with different order IDs but the same procedure share a class and can be counted together. |
| **Promotion gate** | A minimum evidence requirement such as execution count, success rate, query diversity, recency, and application-level review. |
| **Business-outcome evaluator** | A runtime callback that labels the completed workflow using domain facts before storage, promotion, and drift accounting. It does not change the prompt or held-out score. |
| **Distillation** | Instruction synthesis in this guide: the configured LLM turns sampled trajectories into a compact, validated `SKILL.md`. It does not change or fine-tune model weights. |
| **Skill** | A validated instruction document distilled from repeated workflows. It states when to apply, preconditions, procedure, tools, and observed failure modes. |
| **Shadow skill** | A candidate skill stored for inspection but excluded from retrieval until it is explicitly activated. |
| **Canary** | A small, strict post-activation test that must pass before the larger held-out evaluation proceeds. |
| **Held-out case** | A request excluded from seed trajectories and distillation inputs, used only after the skill and scoring rubric are fixed for that execution. |
| **Positive control** | A reference arm expected to succeed if the task is solvable. Here it receives the full SOP directly but no learned skill; it is not part of the primary paired treatment estimate. |
| **Toolbox** | MemoRizz's persistent catalog of tool metadata plus the live Python callables available in the current process. |
| **Skillbox** | The memory partition and manager for versioned skill documents and lifecycle state. |
| **Instruction privilege** | Priority arising from the runtime's role hierarchy or managed instruction placement. It is a property of how content is loaded, not of a filename alone. |
| **Embedding** | A vector representation used for semantic similarity. The same model and dimension must be used for writes and queries. |
| **In-database embedding** | Embedding inference executed by Oracle with `VECTOR_EMBEDDING`, so text need not leave the database process for vectorization. |
| **ONNX** | Open Neural Network Exchange, the portable model format loaded into Oracle by `DBMS_VECTOR.LOAD_ONNX_MODEL`. |
| **Negative transfer** | A retrieved skill hurts a task because it matched superficially but its preconditions did not hold. |
| **Drift / demotion** | A sustained post-promotion regression, stale tool dependency, or applicability problem; demotion removes the skill from active retrieval. |
| **Accuracy** | The fraction of held-out cases with the exact expected tool trajectory and correct business events. |
| **Provider inference latency** | Sum of wall-clock time inside every selected provider `generate` call for one agent run, including network and provider queueing for cloud APIs. |
| **End-to-end latency** | Wall-clock time around `MemAgent.run`: Oracle retrieval and writes, embeddings, every LLM call, tool execution, orchestration, and any network time. |

### Part I Key Takeaways

- Workflows are execution evidence; skills are reviewed instructions compiled from that evidence.
- Instruction priority comes from runtime placement and hierarchy, not from the `SKILL.md` filename.
- Continual learning here updates operational memory and instruction artifacts, not model weights.

**Continue to Part II:** turn this conceptual contract into a reproducible live runtime
whose installed package, database, embeddings, and model can be verified.

## Part II - Establish the Live Runtime

Part I defined what the system is allowed to learn. Part II makes the infrastructure
observable so a provider or setup failure cannot be mistaken for a learning result.

### 0. Prerequisites

Select the intended Jupyter kernel, then run the package-install cell below before any
imports. It installs the newest published MemoRizz release into that kernel:

```bash
conda activate memorizz_local
python -m jupyter kernelspec list
python -m pip install --upgrade --force-reinstall "memorizz[oracle,ollama]"
memorizz setup-oracle
```

The notebook uses `%pip`, so installation targets the active kernel rather than whichever
`pip` happens to be on the shell path. `--upgrade` pulls the newest available release;
`--force-reinstall` replaces an existing editable installation that could still point at
a local checkout. If MemoRizz was already imported in this kernel, restart the kernel after
the install cell and continue from the environment cell.

The environment cell loads `ORACLE_USER`, `ORACLE_PASSWORD`, `ORACLE_DSN`, and provider
credentials from the nearest `.env`; `ORACLE_SCHEMA` is optional. Outputs redact secrets.
The database user needs `CREATE MINING MODEL` and
`EXECUTE ON DBMS_VECTOR`. A fresh MemoRizz Oracle schema defaults to
384-dimensional vector columns, matching the ONNX model used here.

#### Select Either LLM Provider

OpenAI is the default configuration:

```bash
export MEMORIZZ_GUIDE_LLM_PROVIDER=openai
export OPENAI_API_KEY="set-in-a-secret-manager-or-ignored-.env"
export OPENAI_MODEL=gpt-5.6
export OPENAI_REASONING_EFFORT=none
```

Or use Ollama:

```bash
export MEMORIZZ_GUIDE_LLM_PROVIDER=ollama
export OLLAMA_HOST=http://127.0.0.1:11434
export OLLAMA_MODEL=kimi-k2.7-code:cloud
export OLLAMA_THINK=false
```

`kimi-k2.7-code:cloud` is an Ollama Cloud model, not local model weights. The earlier
live attempt reached Ollama but returned subscription HTTP 403, so it could not be used
as an honest Kimi result. `ollama show` only proves a tag is known; verify entitlement
with `ollama run`. A fully local tool-capable fallback such as `qwen2.5:7b` can be
selected with the same environment variables.

OpenAI's current
[model guidance](https://developers.openai.com/api/docs/guides/latest-model) recommends
the GPT-5.6 family and the Responses API for tool workflows. MemoRizz currently uses
Responses for distillation and Chat Completions for its agent tool loop. The OpenAI
provider sends `reasoning_effort=none` for GPT-5.6 function calls on that endpoint.

The experiment makes real model calls and real Oracle writes. Its final teardown removes
rows created by this run but deliberately retains the shared schema and ONNX model.

In [ ]:
# Install the newest published MemoRizz into this notebook's active kernel.
# --force-reinstall replaces any editable checkout-backed installation.
%pip install --upgrade --force-reinstall "memorizz[oracle,ollama]"

In [ ]:
# Live environment configuration: edit defaults here or override them externally.
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv


env_path = find_dotenv(usecwd=True)
ENV_FILE = Path(env_path).resolve() if env_path else None
if ENV_FILE is not None:
    load_dotenv(str(ENV_FILE), override=False)

os.environ.setdefault("MEMORIZZ_GUIDE_LLM_PROVIDER", "openai")
os.environ.setdefault("OPENAI_MODEL", "gpt-5.6")
os.environ.setdefault("OPENAI_REASONING_EFFORT", "none")
os.environ.setdefault("OLLAMA_HOST", "http://127.0.0.1:11434")
os.environ.setdefault("OLLAMA_MODEL", "kimi-k2.7-code:cloud")
os.environ.setdefault("OLLAMA_THINK", "false")
os.environ.setdefault("ORACLE_SCHEMA", os.environ.get("ORACLE_USER", ""))
os.environ.setdefault("MEMORIZZ_DISABLE_CONVERSATION_EMBEDDINGS", "1")

required_keys = ("ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN")
missing_keys = [name for name in required_keys if not os.environ.get(name)]
if missing_keys:
    raise RuntimeError(f"Missing required environment keys: {missing_keys}")

selected_llm_provider = os.environ["MEMORIZZ_GUIDE_LLM_PROVIDER"].strip().lower()
if selected_llm_provider not in {"openai", "ollama"}:
    raise RuntimeError(
        "MEMORIZZ_GUIDE_LLM_PROVIDER must be 'openai' or 'ollama'."
    )
if selected_llm_provider == "openai" and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required when the OpenAI provider is selected.")

print(
    {
        "environment_file": str(ENV_FILE) if ENV_FILE else "<not present>",
        "oracle_user": os.environ["ORACLE_USER"],
        "oracle_password": "<set, redacted>",
        "oracle_dsn": os.environ["ORACLE_DSN"],
        "oracle_schema": os.environ["ORACLE_SCHEMA"],
        "llm_provider": selected_llm_provider,
        "openai_key": "<set, redacted>" if os.environ.get("OPENAI_API_KEY") else "<not set>",
        "openai_model": os.environ["OPENAI_MODEL"],
        "ollama_host": os.environ["OLLAMA_HOST"],
        "ollama_model": os.environ["OLLAMA_MODEL"],
    }
)

In [ ]:
import json
import math
import random
import statistics
import time
import uuid
from collections import defaultdict
from datetime import datetime, timezone
from importlib.metadata import version

import memorizz
from memorizz import MemAgent, get_tool_context
from memorizz.embeddings import get_embedding, get_embedding_manager
from memorizz.enums.memory_type import MemoryType
from memorizz.llms.llm_provider import LLMProvider
from memorizz.llms.ollama import OllamaLLM
from memorizz.llms.openai import OpenAI
from memorizz.long_term.procedural.skillbox import SkillStatus
from memorizz.long_term.procedural.toolbox import Toolbox
from memorizz.long_term.procedural.workflow import (
    aggregate_trajectory_stats,
    canonical_signature,
)
from memorizz.memory_provider.oracle import OracleConfig, OracleProvider


MEMORIZZ_IMPORT = Path(memorizz.__file__).resolve()
MEMORIZZ_VERSION = version("memorizz")
if not any(
    part in {"site-packages", "dist-packages"}
    for part in MEMORIZZ_IMPORT.parts
):
    raise RuntimeError(
        "MemoRizz is not resolving from an installed distribution. "
        f"Run the %pip install cell, restart the kernel, and retry: {MEMORIZZ_IMPORT}"
    )
print(
    {
        "memorizz_version": MEMORIZZ_VERSION,
        "memorizz_import": str(MEMORIZZ_IMPORT),
    }
)


def required_env(name: str) -> str:
    value = os.getenv(name, "").strip()
    if not value:
        raise RuntimeError(f"Set {name} before running this notebook.")
    return value


def env_flag(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return raw.strip().lower() in {"1", "true", "yes", "on"}


RUN_ID = os.getenv("MEMORIZZ_GUIDE_RUN_ID", uuid.uuid4().hex[:10])
LEARNING_AGENT_ID = f"continual-learning-skill-{RUN_ID}"
CAPTURE_AGENT_ID = f"continual-learning-capture-{RUN_ID}"
EXPLICIT_SOP_AGENT_ID = f"continual-learning-explicit-sop-{RUN_ID}"
DEMO_USER_ID = f"guide-user-{RUN_ID}"
LLM_PROVIDER = os.getenv("MEMORIZZ_GUIDE_LLM_PROVIDER", "openai").strip().lower()
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.6")
OPENAI_REASONING_EFFORT = os.getenv("OPENAI_REASONING_EFFORT", "none")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "kimi-k2.7-code:cloud")
OLLAMA_HOST = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434")

print(
    {
        "run_id": RUN_ID,
        "learning_agent": LEARNING_AGENT_ID,
        "capture_only_agent": CAPTURE_AGENT_ID,
        "explicit_sop_agent": EXPLICIT_SOP_AGENT_ID,
        "llm_provider": LLM_PROVIDER,
        "model": OPENAI_MODEL if LLM_PROVIDER == "openai" else OLLAMA_MODEL,
    }
)

### 1. Oracle AI Database as Memory and Embedding Provider

`OracleConfig.in_database_embedding` defaults to `True`. With no explicit external
embedding provider, MemoRizz:

1. checks `USER_MINING_MODELS` for `ALL_MINILM_L12_V2`;
2. downloads Oracle's augmented ONNX artifact, or reads `onnx_path`;
3. loads it with the BLOB overload of `DBMS_VECTOR.LOAD_ONNX_MODEL`; and
4. embeds writes and queries with `VECTOR_EMBEDDING(... USING ... AS DATA)`.

The default is the 384-dimensional `all-MiniLM-L12-v2` artifact used by Oracle's
[`total_recall_agent_harness.ipynb`](https://github.com/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/agent_harness/total_recall_agent_harness.ipynb).
Oracle documents
[`DBMS_VECTOR.LOAD_ONNX_MODEL`](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/load_onnx_model-procedure.html)
and the
[in-database embedding quick start](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/sql-quick-start-using-vector-embedding-model-uploaded-database.html).

Set `in_database_embedding=False` and provide `embedding_provider=...` only when an
external embedding service is intentional.

In [ ]:
oracle_user = required_env("ORACLE_USER")
oracle_embedding_config = {
    "model": os.getenv("ORACLE_EMBEDDING_MODEL", "ALL_MINILM_L12_V2"),
    "dimensions": int(os.getenv("ORACLE_EMBEDDING_DIM", "384")),
    "install_if_missing": True,
}

if os.getenv("ORACLE_EMBEDDING_ONNX_PATH"):
    oracle_embedding_config["onnx_path"] = os.environ[
        "ORACLE_EMBEDDING_ONNX_PATH"
    ]
if os.getenv("ORACLE_EMBEDDING_ONNX_URL"):
    oracle_embedding_config["onnx_url"] = os.environ[
        "ORACLE_EMBEDDING_ONNX_URL"
    ]

provider = OracleProvider(
    OracleConfig(
        user=oracle_user,
        password=required_env("ORACLE_PASSWORD"),
        dsn=required_env("ORACLE_DSN"),
        schema=os.getenv("ORACLE_SCHEMA") or oracle_user,
        lazy_vector_indexes=True,
        in_database_embedding=True,
        embedding_config=oracle_embedding_config,
    )
)

embedding_info = get_embedding_manager().get_provider_info()
assert embedding_info["provider"] == "oracle_in_database"
assert embedding_info["dimensions"] == 384
vector_schema_dimensions = provider.get_vector_schema_dimensions()
if not vector_schema_dimensions or set(vector_schema_dimensions.values()) != {384}:
    raise RuntimeError(
        f"Oracle schema is not uniformly 384-dimensional: {vector_schema_dimensions}"
    )
print(
    json.dumps(
        {
            "embedding_provider": embedding_info,
            "schema_vector_dimensions": vector_schema_dimensions,
        },
        indent=2,
    )
)

In [ ]:
probe_text = "Resolve a refund and send its customer receipt."
probe_vector = get_embedding(probe_text)

print(
    {
        "text": probe_text,
        "dimensions": len(probe_vector),
        "first_5_values": [round(value, 5) for value in probe_vector[:5]],
        "inference_location": "Oracle AI Database",
    }
)

Every procedural object below uses the global Oracle embedding adapter. That
includes Toolbox metadata, captured workflows, distilled skills, and semantic retrieval
queries. This avoids mixing vector spaces and keeps embedding inference close to the
memory rows.

### 2. A Real, Switchable LLM Provider

There is no scripted LLM in this guide. The selected provider decides when to call the
live tools, produces final answers, distills repeated trajectories into `SKILL.md`, and
acts as the validation judge.

The preflight makes a real request before Oracle rows are created. It catches API key,
subscription, model access, missing Ollama model, and daemon failures before an
infrastructure error can be mistaken for a continual-learning result.

In [ ]:
def make_llm() -> LLMProvider:
    if LLM_PROVIDER == "openai":
        return OpenAI(
            model=OPENAI_MODEL,
            reasoning_effort=OPENAI_REASONING_EFFORT,
        )
    return OllamaLLM(
        model=OLLAMA_MODEL,
        host=OLLAMA_HOST,
        temperature=0,
        seed=7,
        num_predict=int(os.getenv("OLLAMA_NUM_PREDICT", "2048")),
        context_window_tokens=int(os.getenv("OLLAMA_CONTEXT_WINDOW", "32768")),
        timeout=600,
        think=env_flag("OLLAMA_THINK", False),
    )


learning_llm = make_llm()
print(learning_llm.get_config())

preflight_started = time.perf_counter()
try:
    preflight_response = learning_llm.generate_text(
        "Reply with exactly READY.",
        instructions="This is a connectivity preflight. Return READY only.",
    )
except Exception as exc:
    selected_model = OPENAI_MODEL if LLM_PROVIDER == "openai" else OLLAMA_MODEL
    raise RuntimeError(
        f"{LLM_PROVIDER} model {selected_model!r} is not callable: {exc}"
    ) from exc
if "READY" not in preflight_response.strip().upper():
    raise RuntimeError(f"Unexpected model preflight response: {preflight_response!r}")
print(
    {
        "model_preflight": preflight_response[:80],
        "latency_ms": round((time.perf_counter() - preflight_started) * 1000, 1),
    }
)

### Part II Key Takeaways

- The notebook upgrades and imports the published MemoRizz distribution in the active kernel.
- Oracle owns both memory persistence and 384-dimensional ONNX embedding inference.
- A real provider preflight must succeed before the experiment creates benchmark rows.

**Continue to Part III:** use the verified runtime to create real tool executions and
retain only trajectories that satisfy the independent business rubric.

---

## Part III - Capture Real Agent Experience

With the runtime established, the guide now moves from infrastructure to evidence. The
next two sections define the live tool surface and capture comparable successful runs.

### 3. Real Functions in a Real MemoRizz Toolbox

This benchmark tests retention of an organization-specific SOP:

1. retrieve the current order;
2. issue a refund only when the order is completed; and
3. send a receipt after the refund succeeds.

The receipt step is intentionally a local operating rule, not a requirement stated in
the production prompt. That creates legitimate headroom for learning: the capture-only
arm has passive trajectory records, while the continual-learning arm receives a
distilled instruction when it is relevant.

The tools read private `run_id` and arm labels from `tool_context`; neither appears in
the tool arguments or model-visible tool results. Every returned result is checked for
those forbidden fields. Separate in-memory business events retain the private fields so
the grader remains independent of the model's prose.

In [ ]:
ORDERS = {
    **{
        f"R-{number}": {
            "order_id": f"R-{number}",
            "status": "completed",
            "amount": 49.0,
        }
        for number in range(1001, 1031)
    },
    "R-1098": {
        "order_id": "R-1098",
        "status": "processing",
        "amount": 79.0,
    },
    "R-1099": {
        "order_id": "R-1099",
        "status": "pending",
        "amount": 29.0,
    },
}
LOOKED_UP = set()
REFUNDED = set()
REFUND_EVENTS = []
RECEIPT_EVENTS = []
MODEL_VISIBLE_TOOL_RESULTS = []
FORBIDDEN_MODEL_FIELDS = {"run_id", "arm"}


def _tool_run_key(order_id: str):
    context = get_tool_context()
    return (str(context.get("run_id") or "manual"), order_id)


def _model_visible_tool_result(result: dict) -> dict:
    """Record and return only fields the LLM is allowed to observe."""
    exposed = dict(result)
    leaked = FORBIDDEN_MODEL_FIELDS.intersection(exposed)
    if leaked:
        raise RuntimeError(f"Internal experiment fields leaked to the model: {leaked}")
    MODEL_VISIBLE_TOOL_RESULTS.append(exposed)
    return exposed


def lookup_order(order_id: str) -> dict:
    """Retrieve the current record for one order."""
    order = ORDERS.get(order_id)
    LOOKED_UP.add(_tool_run_key(order_id))
    if order is None:
        return _model_visible_tool_result(
            {"ok": False, "order_id": order_id, "error": "order_not_found"}
        )
    return _model_visible_tool_result({"ok": True, **order})


def issue_refund(order_id: str) -> dict:
    """Apply a refund after this run has retrieved an eligible order."""
    context = get_tool_context()
    run_key = _tool_run_key(order_id)
    order = ORDERS.get(order_id)
    prerequisite_met = run_key in LOOKED_UP
    accepted = bool(
        prerequisite_met and order and order.get("status") == "completed"
    )
    if accepted:
        REFUNDED.add(run_key)
    event = {
        "run_id": run_key[0],
        "arm": context.get("arm"),
        "order_id": order_id,
        "accepted": accepted,
        "reason": (
            "refunded"
            if accepted
            else "lookup_required"
            if not prerequisite_met
            else f"status_{order.get('status') if order else 'missing'}"
        ),
    }
    REFUND_EVENTS.append(event)
    return _model_visible_tool_result(
        {
            "ok": accepted,
            "order_id": order_id,
            "accepted": accepted,
            "reason": event["reason"],
        }
    )


def send_refund_receipt(order_id: str) -> dict:
    """Send a customer receipt for a refund that already succeeded."""
    context = get_tool_context()
    run_key = _tool_run_key(order_id)
    accepted = run_key in REFUNDED
    event = {
        "run_id": run_key[0],
        "arm": context.get("arm"),
        "order_id": order_id,
        "accepted": accepted,
        "reason": "receipt_sent" if accepted else "refund_required",
    }
    RECEIPT_EVENTS.append(event)
    return _model_visible_tool_result(
        {
            "ok": accepted,
            "order_id": order_id,
            "accepted": accepted,
            "reason": event["reason"],
        }
    )

#### Setup Notes for AI Developers

Use a dedicated schema for a reproducible live run. Run the grants as an Oracle
administrator, replacing `MEMORIZZ_CL_GUIDE` with the chosen schema:

```sql
GRANT CREATE SESSION, CREATE TABLE, CREATE VIEW, CREATE SEQUENCE,
      CREATE TRIGGER, CREATE MINING MODEL, UNLIMITED TABLESPACE
  TO MEMORIZZ_CL_GUIDE;
GRANT EXECUTE ON SYS.DBMS_VECTOR TO MEMORIZZ_CL_GUIDE;
```

Vector dimensions are part of Oracle table DDL. Do not point this 384-dimensional guide
at an older 256-dimensional MemoRizz schema; migrate it deliberately or use a fresh
schema. The provider can install the ONNX model, but it should not silently reshape
existing vector columns.

Before executing all cells, verify the intended runtime:

```bash
conda activate memorizz_local
python -m jupyter kernelspec list
ollama run "${OLLAMA_MODEL:-kimi-k2.7-code:cloud}" "Reply with READY"
```

The environment cell uses `load_dotenv(..., override=False)`. CI, a notebook runner, or
a secret manager can inject credentials without being overwritten by `.env`. Never put
an API key or database password directly in notebook source. The import cell prints the
installed MemoRizz version and location and stops if it does not resolve from
`site-packages` or `dist-packages`.

Oracle enforces parent-child integrity for agent-scoped procedural memory. This guide
therefore calls `MemAgent.save()` before inserting Toolbox rows for an agent.

#### Live-Run Diagnostics for AI Developers

Treat the first failing layer as the diagnosis; do not compensate by inserting synthetic
workflows or lowering promotion gates:

1. **Wrong package:** rerun the `%pip install --upgrade --force-reinstall` cell, restart
   the kernel, and confirm the import cell prints `site-packages` or `dist-packages`.
2. **Kimi HTTP 403:** the Ollama Cloud tag is known, but the signed-in account cannot
   invoke it. Obtain the required entitlement, select a local Ollama model, or switch
   `MEMORIZZ_GUIDE_LLM_PROVIDER=openai`.
3. **OpenAI function-tool HTTP 400:** GPT-5.6 on Chat Completions may require
   `OPENAI_REASONING_EFFORT=none` for function tools. Current MemoRizz releases expose
   that setting.
4. **No workflow rows:** inspect `TOOL_LOG` and `WORKFLOW_MEMORY`. Zero rows normally
   means the model emitted no structured tool call. Check schemas and keep the
   experiment's tool surface controlled.
5. **Model stops after lookup despite a returned status:** inspect the current-turn tool placeholder. It must retain bounded scalar fields such as `ok=True`, `status=completed`, `accepted`, and `reason`; retaining only `ok=True` removes the facts needed for the next decision. Upgrade MemoRizz before diagnosing provider behavior.
6. **Oracle promotion reports no timestamps or query diversity:** verify the current
   schema has `USER_QUERY`, `CREATED_AT`, and `UPDATED_AT`.
7. **Vector dimension errors:** table vector widths and ONNX output dimensions must
   agree. Use a fresh 384-dimensional demo schema or migrate and re-embed explicitly.

This notebook passes the same domain rubric to every agent through
`workflow_outcome_evaluator`. MemoRizz applies it before storage, promotion, and drift
accounting, closing the gap where execution-level `outcome=success` could hide a rejected
business action. The callback does not alter prompts, tool results, or the independent
held-out grader. Production applications should provide an equally strict evaluator.

In [ ]:
COACHING_INSTRUCTION = """
You are collecting verified demonstrations of the company refund SOP.

For every refund request:
1. Call lookup_order exactly once.
2. If and only if its status is completed, call issue_refund exactly once.
3. If the refund succeeds, call send_refund_receipt exactly once.
4. For an ineligible or missing order, stop after lookup and explain why.

Never claim success unless all required tool results report ok=true.
Keep the final answer concise.
""".strip()

PRODUCTION_INSTRUCTION = """
You operate order support. Complete the user's requested task with the available tools.
Verify current data before state-changing actions, obey tool eligibility results, and
never claim an action succeeded unless a tool confirms it. Keep the final answer concise.
""".strip()

PROMOTION_CONFIG = {
    "min_executions": 5,
    "min_success_rate": 0.80,
    "min_distinct_queries": 5,
    "max_days_since_last_seen": 30,
    "distill_sample_size": 5,
    "include_failure_samples": 2,
    "require_shadow": True,
    "retrieval_min_similarity": 0.35,
    "max_skills_in_context": 1,
    "include_exemplar": False,
    "promotion_every_n_runs": 0,
    "drift_window_activations": 10,
    "min_activations_before_drift_check": 5,
    "demotion_success_delta": 0.25,
}


REFUND_INTENT_TERMS = (
    "refund",
    "return the payment",
    "return payment",
    "money back",
    "reverse the charge",
    "reverse the completed purchase",
)


def workflow_business_outcome(workflow) -> bool:
    """Symmetric domain rubric used only for workflow outcome accounting."""
    path = [
        unit.get("tool")
        for unit in canonical_signature(workflow.steps or {})
    ]
    query = str(workflow.user_query or "").lower()
    refund_intent = any(term in query for term in REFUND_INTENT_TERMS)
    lookup_status = None
    for step_name, step in (workflow.steps or {}).items():
        if "lookup_order" not in step_name or not isinstance(step, dict):
            continue
        result = step.get("result")
        if isinstance(result, dict):
            lookup_status = result.get("status")
        break

    eligible_refund = refund_intent and lookup_status == "completed"
    expected_path = (
        ["lookup_order", "issue_refund", "send_refund_receipt"]
        if eligible_refund
        else ["lookup_order"]
    )
    run_id = str(workflow.memory_id)
    refunds = [event for event in REFUND_EVENTS if event["run_id"] == run_id]
    receipts = [event for event in RECEIPT_EVENTS if event["run_id"] == run_id]
    side_effects_correct = (
        len(refunds) == 1
        and refunds[0]["accepted"]
        and len(receipts) == 1
        and receipts[0]["accepted"]
        if eligible_refund
        else len(refunds) == 0 and len(receipts) == 0
    )
    return path == expected_path and side_effects_correct


def make_agent(
    agent_id: str,
    model: LLMProvider,
    continual_learning: bool,
    instruction: str,
) -> MemAgent:
    memory_types = [MemoryType.WORKFLOW_MEMORY]
    if continual_learning:
        memory_types.append(MemoryType.SKILLBOX)

    agent = MemAgent(
        model=model,
        llm_config=model.get_config(),
        instruction=instruction,
        memory_provider=provider,
        memory_types=memory_types,
        agent_id=agent_id,
        max_steps=10,
        semantic_cache=False,
        continual_learning=continual_learning,
        continual_learning_config=(
            PROMOTION_CONFIG if continual_learning else None
        ),
        workflow_outcome_evaluator=workflow_business_outcome,
        automations_enabled=False,
    )

    for tool_name in list(agent.tool_manager.list_tools()):
        agent.tool_manager.remove_tool(tool_name)

    agent.save()
    return agent


learning_agent = make_agent(
    LEARNING_AGENT_ID,
    learning_llm,
    continual_learning=True,
    instruction=COACHING_INSTRUCTION,
)
capture_llm = make_llm()
capture_agent = make_agent(
    CAPTURE_AGENT_ID,
    capture_llm,
    continual_learning=True,
    instruction=COACHING_INSTRUCTION,
)
explicit_sop_llm = make_llm()
explicit_sop_agent = make_agent(
    EXPLICIT_SOP_AGENT_ID,
    explicit_sop_llm,
    continual_learning=True,
    instruction=COACHING_INSTRUCTION,
)
print(
    {
        "learning_parent": learning_agent.agent_id,
        "capture_only_parent": capture_agent.agent_id,
        "explicit_sop_parent": explicit_sop_agent.agent_id,
    }
)

In [ ]:
toolbox = Toolbox(
    memory_provider=provider,
    llm_provider=learning_llm,
    agent_id=LEARNING_AGENT_ID,
)
TOOL_IDS = [
    toolbox.register_tool(lookup_order),
    toolbox.register_tool(issue_refund),
    toolbox.register_tool(send_refund_receipt),
]

available = toolbox.list_available_tools()
related_candidates = toolbox.get_most_similar_tools(
    "Resolve an order refund and provide its receipt",
    limit=12,
)
related = []
seen_tool_names = set()
for candidate in related_candidates:
    name = candidate.get("name")
    if not name or name in seen_tool_names:
        continue
    seen_tool_names.add(name)
    related.append(candidate)
    if len(related) == 3:
        break
print("Persisted live tools:", [row.get("name") for row in available])
print(
    "Oracle vector matches:",
    [(row.get("name"), round(float(row.get("score", 0.0)), 3)) for row in related],
)

for agent in (learning_agent, capture_agent, explicit_sop_agent):
    loaded = agent.tool_manager.initialize_from_toolbox(toolbox)
    if loaded != 3:
        raise RuntimeError(
            f"Expected three live Toolbox functions for {agent.agent_id}, loaded {loaded}."
        )
    print(agent.agent_id, agent.tool_manager.list_tools())

`MemAgent.save()` persists each Oracle parent row before agent-scoped procedural
rows are inserted. `Toolbox.register_tool` stores metadata while retaining the exact
function objects in this process, and `ToolManager.initialize_from_toolbox` loads those
callables into all three agents.

The two primary arms use the same model family, coaching instruction, tool schemas, seed
queries, business-outcome callback, and real Oracle memory. After seed collection they
both switch to the same production instruction. The treatment difference is narrow:

- **capture_only:** keeps the successful workflow rows as passive evidence;
- **continual_learning:** distills the same trajectory class, reviews it in shadow
  state, activates it, and retrieves it for relevant held-out requests.
- **explicit_sop_control:** receives the full SOP directly, has no learned skill, and is
  evaluated only as a positive reference; it is not part of the primary paired contrast.

MemoRizz does not automatically inject raw workflow rows during pre-inference. This is
the product behavior being tested: whether compiling passive evidence into a selected
instruction improves execution.

### 4. Capture and Business-Grade Repeated Trajectories

The loop interleaves the two primary arms to reduce provider-time bias. A seed counts
only when
the stored workflow is exactly
`lookup_order -> issue_refund -> send_refund_receipt` **and** both business events were
accepted. The same rubric labels the stored workflow before promotion sees it;
execution-level success alone is insufficient.

Order IDs are removed from trajectory identity; tool order, argument-key shape, and
error class remain. The notebook stops visibly if either arm cannot produce five
verified demonstrations.

In [ ]:
SEED_REQUESTS = [
    ("R-1001", "Please resolve the refund for order R-1001."),
    ("R-1002", "Can you return the payment for R-1002?"),
    ("R-1003", "I need my money back on R-1003."),
    ("R-1004", "Process the eligible refund for order R-1004."),
    ("R-1005", "Reverse the completed purchase R-1005."),
    ("R-1006", "Handle R-1006's refund request."),
    ("R-1007", "Please complete the return payment for R-1007."),
    ("R-1008", "Refund R-1008 after checking the order."),
]
EXPECTED_SUCCESS_PATH = [
    "lookup_order",
    "issue_refund",
    "send_refund_receipt",
]


def agent_workflows(agent_id: str):
    rows = provider.list_all(memory_store_type=MemoryType.WORKFLOW_MEMORY) or []
    return [row for row in rows if row.get("agent_id") == agent_id]


def tool_path(workflow_doc: dict):
    signature = workflow_doc.get("canonical_signature")
    if not signature:
        signature = canonical_signature(workflow_doc.get("steps") or {})
    return [unit.get("tool") for unit in signature]


def latest_workflow(agent_id: str, memory_id: str):
    matches = [
        row
        for row in agent_workflows(agent_id)
        if row.get("memory_id") == memory_id
    ]
    return matches[-1] if matches else None


def run_events(run_id: str):
    refunds = [event for event in REFUND_EVENTS if event["run_id"] == run_id]
    receipts = [event for event in RECEIPT_EVENTS if event["run_id"] == run_id]
    return refunds, receipts


def grade_run(agent_id: str, run_id: str, eligible: bool):
    workflow_doc = latest_workflow(agent_id, run_id)
    actual_path = tool_path(workflow_doc) if workflow_doc else []
    refunds, receipts = run_events(run_id)
    expected_path = EXPECTED_SUCCESS_PATH if eligible else ["lookup_order"]
    business_correct = (
        len(refunds) == 1
        and refunds[0]["accepted"]
        and len(receipts) == 1
        and receipts[0]["accepted"]
        if eligible
        else len(refunds) == 0 and len(receipts) == 0
    )
    correct = actual_path == expected_path and business_correct
    recorded_business_success = bool(
        workflow_doc and str(workflow_doc.get("outcome")) == "success"
    )
    return {
        "correct": correct,
        "recorded_business_success": recorded_business_success,
        "outcome_agrees_with_grade": recorded_business_success == correct,
        "expected_path": expected_path,
        "actual_path": actual_path,
        "refunds": refunds,
        "receipts": receipts,
        "workflow": workflow_doc,
    }


seed_successes = {"capture_only": [], "continual_learning": []}
seed_agents = {
    "capture_only": capture_agent,
    "continual_learning": learning_agent,
}

for index, (order_id, query) in enumerate(SEED_REQUESTS, start=1):
    arm_order = (
        ["capture_only", "continual_learning"]
        if index % 2
        else ["continual_learning", "capture_only"]
    )
    for arm in arm_order:
        if len(seed_successes[arm]) >= PROMOTION_CONFIG["min_executions"]:
            continue
        agent = seed_agents[arm]
        run_id = f"{RUN_ID}-seed-{index}-{arm}"
        response = agent.run(
            query,
            memory_id=run_id,
            thread_id=str(uuid.uuid4()),
            user_id=DEMO_USER_ID,
            tool_context={"run_id": run_id, "arm": f"{arm}_seed"},
        )
        grade = grade_run(agent.agent_id, run_id, eligible=True)
        if grade["correct"]:
            seed_successes[arm].append(
                {"order_id": order_id, "query": query, "run_id": run_id}
            )
        print(
            arm,
            index,
            {
                "correct": grade["correct"],
                "path": grade["actual_path"],
                "response": response[:100],
            },
        )
    if all(
        len(rows) >= PROMOTION_CONFIG["min_executions"]
        for rows in seed_successes.values()
    ):
        break

for arm, rows in seed_successes.items():
    if len(rows) < PROMOTION_CONFIG["min_executions"]:
        raise RuntimeError(
            f"{arm} produced only {len(rows)} business-correct seed trajectories."
        )

print("Verified seed successes:", {key: len(value) for key, value in seed_successes.items()})

In [ ]:
learning_matches = [
    row
    for row in agent_workflows(LEARNING_AGENT_ID)
    if tool_path(row) == EXPECTED_SUCCESS_PATH
    and str(row.get("outcome", "success")) == "success"
]
target_hash = learning_matches[0]["canonical_hash"]
stats = aggregate_trajectory_stats(provider, agent_id=LEARNING_AGENT_ID)
target_stats = next(item for item in stats if item.canonical_hash == target_hash)

print(
    {
        "canonical_hash": target_hash,
        "canonical_signature": learning_matches[0]["canonical_signature"],
        "executions": target_stats.executions,
        "success_rate": target_stats.success_rate,
        "business_verified_executions": len(seed_successes["continual_learning"]),
        "distinct_queries": target_stats.distinct_query_count,
        "first_seen": target_stats.first_seen.isoformat(),
        "last_seen": target_stats.last_seen.isoformat(),
    }
)

### Part III Key Takeaways

- Every arm uses the same real Toolbox functions and Oracle-backed workflow memory.
- A trajectory counts only when its exact tool path and business side effects are correct.
- Canonicalization groups the procedure while excluding order-specific values from identity.

**Continue to Part IV:** convert the repeated, business-graded trajectory class into a
reviewed skill, then prove it is safe enough to remain active.

---

## Part IV - Distill and Govern the Skill

Part III produced evidence, not authority. Part IV applies promotion gates, validation,
canaries, and monitoring before that evidence can influence later model calls.

### 5. Distill in Shadow, Review, and Activate

Frequency alone is insufficient. MemoRizz checks executions, success rate, distinct
queries, and recency. The selected model then distills sampled real runs into constrained
`SKILL.md`; machine validation checks frontmatter, tool existence, undeclared calls,
content length, and literal-value leakage across both successful and failed samples. A
second model call judges whether the document could reproduce successes while avoiding
or correctly handling observed failures.

`require_shadow=True` prevents immediate retrieval. This notebook then performs an
application-level review: all expected tools must be declared, the receipt step must be
present, and no training order ID may leak into the instruction. Only then is the
candidate activated.

In [ ]:
manager = learning_agent.continual_learning_manager
promotion_report = manager.promote_class(target_hash)
print(json.dumps(promotion_report.to_dict(), indent=2))

if not promotion_report.promoted:
    raise RuntimeError(
        "Real-model distillation did not pass validation. "
        "Inspect promotion_report.rejected before retrying."
    )

promoted_skill_id = promotion_report.promoted[0]
shadow_skill = manager.skillbox.get_skill_by_id(promoted_skill_id)
assert shadow_skill is not None
assert shadow_skill.status == SkillStatus.SHADOW

expected_tools = {
    "lookup_order",
    "issue_refund",
    "send_refund_receipt",
}
declared_tools = set(shadow_skill.tools_used)
leaked_ids = [
    row["order_id"]
    for row in seed_successes["continual_learning"]
    if row["order_id"] in shadow_skill.content
]
review = {
    "status_before_review": shadow_skill.status.value,
    "declared_tools": sorted(declared_tools),
    "all_expected_tools_declared": expected_tools <= declared_tools,
    "receipt_step_present": "send_refund_receipt" in shadow_skill.content,
    "training_ids_leaked": leaked_ids,
}
print(review)
print(shadow_skill.content)

if not (
    review["all_expected_tools_declared"]
    and review["receipt_step_present"]
    and not review["training_ids_leaked"]
):
    raise RuntimeError("Shadow skill failed the application-level review.")

if not manager.activate_skill(promoted_skill_id):
    raise RuntimeError("Reviewed shadow skill could not be activated.")
promoted_skill = manager.skillbox.get_skill_by_id(promoted_skill_id)
assert promoted_skill is not None
assert promoted_skill.status == SkillStatus.ACTIVE

# The primary arms now receive the same production instruction. The positive
# control deliberately retains the explicit SOP and never receives a learned skill.
learning_agent.instruction = PRODUCTION_INSTRUCTION
capture_agent.instruction = PRODUCTION_INSTRUCTION

primary_prompts_identical = (
    learning_agent._build_system_prompt() == capture_agent._build_system_prompt()
)
primary_tool_schemas_identical = (
    learning_agent._build_llm_tools() == capture_agent._build_llm_tools()
)
control_skill_counts = {
    "capture_only": len(
        capture_agent.continual_learning_manager.skillbox.list_skills()
    ),
    "explicit_sop_control": len(
        explicit_sop_agent.continual_learning_manager.skillbox.list_skills()
    ),
}
if not primary_prompts_identical or not primary_tool_schemas_identical:
    raise RuntimeError("Primary arms differ outside learned-skill activation.")
if any(control_skill_counts.values()):
    raise RuntimeError(f"A control arm unexpectedly contains a skill: {control_skill_counts}")

print(
    {
        "status_after_review": promoted_skill.status.value,
        "primary_system_prompts_identical": primary_prompts_identical,
        "primary_tool_schemas_identical": primary_tool_schemas_identical,
        "control_skill_counts": control_skill_counts,
    }
)

In [ ]:
scored_skills = manager.retrieve_skills_for_query(
    "Please resolve the refund for completed order R-1010."
)
print(
    [
        {
            "skill_id": item.skill.skill_id,
            "name": item.skill.name,
            "similarity": round(item.similarity, 3),
            "preconditions": item.skill.preconditions,
        }
        for item in scored_skills
    ]
)
print(manager.format_skills_prompt_section(scored_skills))

### 6. Canary the Learned Skill

Passing review makes a candidate eligible for activation; it does not establish useful
behavior. The canaries therefore test both the learned procedure and its stop conditions
before the larger comparison begins.

Activation is not a claim of improvement. Two unseen canaries run first:

- a completed refund must perform all three steps and produce accepted refund and
  receipt events;
- a processing order must stop after lookup without attempting either side effect.

Both must activate the learned skill and pass the strict rubric. A failure immediately
demotes the skill and stops the notebook before the larger evaluation.

In [ ]:
CANARY_CASES = [
    {
        "case_id": "canary_completed",
        "query": "Please resolve the refund for completed order R-1010.",
        "eligible": True,
    },
    {
        "case_id": "canary_processing_guardrail",
        "query": "Refund R-1098 only if its current state permits it.",
        "eligible": False,
    },
]
canary_rows = []

for case in CANARY_CASES:
    run_id = f"{RUN_ID}-{case['case_id']}"
    response = learning_agent.run(
        case["query"],
        memory_id=run_id,
        thread_id=str(uuid.uuid4()),
        user_id=DEMO_USER_ID,
        tool_context={"run_id": run_id, "arm": "continual_learning_canary"},
    )
    grade = grade_run(LEARNING_AGENT_ID, run_id, eligible=case["eligible"])
    skill_activated = bool(
        grade["workflow"] and grade["workflow"].get("skills_activated")
    )
    row = {
        "case_id": case["case_id"],
        "correct": grade["correct"],
        "path": grade["actual_path"],
        "skill_activated": skill_activated,
        "response": response,
    }
    canary_rows.append(row)
    print(row)

if not all(row["correct"] and row["skill_activated"] for row in canary_rows):
    manager.demote_skill(
        promoted_skill_id,
        reason="failed strict notebook canary",
    )
    raise RuntimeError("The learned skill failed canary and was demoted.")

### 7. Monitoring, Negative Transfer, and Demotion

Canaries establish immediate readiness. Monitoring extends that decision across future
activations, where drift, tool changes, or broader retrieval can reveal negative transfer.

Promotion is not permanent authority. Every post-promotion workflow stores
`skills_activated`; `SkillMonitor` updates activation, success, and deviation
statistics. Once the minimum activation count is met, a rolling success rate that falls
more than `demotion_success_delta` below the frozen baseline can demote the skill. A
promotion cycle also performs a staleness sweep, so a skill that references a removed
tool is demoted.

MemoRizz's built-in monitor now receives the notebook's business outcome because the
`workflow_outcome_evaluator` runs before workflow persistence and attribution. The exact
held-out grader remains separate, so evaluator/grade agreement can be checked rather
than assumed.

In [ ]:
refreshed_skill = manager.skillbox.get_skill_by_id(promoted_skill_id)
print(
    {
        "status": refreshed_skill.status.value,
        "promotion_baseline": refreshed_skill.baseline,
        "business_outcome_live_stats": refreshed_skill.stats,
        "business_canaries_passed": sum(row["correct"] for row in canary_rows),
        "business_canaries_total": len(canary_rows),
        "drift_window": manager.config.drift_window_activations,
        "minimum_activations": manager.config.min_activations_before_drift_check,
        "allowed_success_drop": manager.config.demotion_success_delta,
    }
)

### Part IV Key Takeaways

- Promotion requires repeated evidence, query diversity, model validation, and review.
- Canaries test both task completion and guardrail behavior before held-out evaluation.
- Attribution, drift monitoring, staleness checks, and demotion keep activation reversible.

**Continue to Part V:** hold the runtime and tool surface fixed, vary learned-skill
availability, and measure both execution quality and cost.

---

## Part V - Measure What Changed

With a reviewed skill active and a rollback path defined, the final empirical question is
whether the compiled instruction changes held-out behavior relative to passive capture.

### 8. Paired Accuracy and Latency Experiment

The held-out set has ten unseen requests:

- six paraphrased eligible refunds;
- two status-only controls;
- one pending-order guardrail; and
- one missing-order guardrail.

Each case runs once through all three arms with a fresh memory and thread. Arm order is
randomized with a fixed seed. Accuracy is strict: eligible refunds require exactly
`lookup_order -> issue_refund -> send_refund_receipt` plus accepted business events;
controls require exactly `lookup_order` and no refund or receipt attempt.

`InferenceMeter` wraps every provider `generate` call, including post-tool follow-ups.
End-to-end latency wraps all of `MemAgent.run`. Cloud-provider inference time therefore
includes network and queueing; the non-inference difference includes Oracle,
in-database embeddings, Python tools, persistence, and orchestration.

The primary paired estimate compares `capture_only` with `continual_learning`. All arms
enable the same dormant continual-learning scaffolding, but only the treatment arm gets a
promoted skill. The notebook asserts that the primary system prompts and tool schemas are
byte-for-byte identical before evaluation; their per-turn difference is the retrieved
skill. `explicit_sop_control` is a no-promoted-skill positive reference that keeps the full
receipt SOP in its prompt. If the learned arm approaches that control, the skill is
recovering a taught instruction, not discovering a new policy.

This is a purpose-built smoke evaluation, not a production claim. Ten cases are enough to
expose the earlier three-case instability, but not enough to establish a general effect
across domains, prompts, providers, or model versions.

In [ ]:
class InferenceMeter:
    """Time all non-streaming generate calls made by one LLM provider."""

    def __init__(self, model: LLMProvider):
        self.model = model
        self.events = []
        self._original = None

    def __enter__(self):
        self._original = self.model.generate

        def timed_generate(*args, **kwargs):
            messages = args[0] if args else kwargs.get("messages", [])
            rendered_messages = json.dumps(
                messages,
                default=str,
                ensure_ascii=False,
            ).lower()
            audit = {
                "arm_label_visible": any(
                    label in rendered_messages
                    for label in (
                        "capture_only",
                        "continual_learning",
                        "explicit_sop_control",
                    )
                ),
                "run_identifier_visible": RUN_ID.lower() in rendered_messages,
                "raw_workflow_context_present": any(
                    marker in rendered_messages
                    for marker in (
                        "• [workflow",
                        "• [procedural_workflow",
                    )
                ),
                "skill_context_present": "── skill:" in rendered_messages,
                "training_identifier_visible": any(
                    row["order_id"].lower() in rendered_messages
                    for row in seed_successes["continual_learning"]
                ),
            }
            started = time.perf_counter()
            try:
                return self._original(*args, **kwargs)
            finally:
                elapsed = time.perf_counter() - started
                usage = self.model.get_last_usage() or {}
                self.events.append(
                    {"seconds": elapsed, "usage": dict(usage), **audit}
                )

        self.model.generate = timed_generate
        return self

    def __exit__(self, exc_type, exc, traceback):
        self.model.generate = self._original

    def reset(self):
        self.events.clear()

    @property
    def inference_seconds(self):
        return sum(event["seconds"] for event in self.events)

    @property
    def usage(self):
        totals = defaultdict(int)
        for event in self.events:
            for key, value in event["usage"].items():
                totals[key] += int(value or 0)
        return dict(totals)


def run_measured(
    arm: str,
    agent: MemAgent,
    meter: InferenceMeter,
    case: dict,
):
    run_id = f"{RUN_ID}-eval-{case['case_id']}-{arm}"
    meter.reset()
    started = time.perf_counter()
    response = agent.run(
        case["query"],
        memory_id=run_id,
        thread_id=str(uuid.uuid4()),
        user_id=DEMO_USER_ID,
        tool_context={"run_id": run_id, "arm": arm},
    )
    end_to_end = time.perf_counter() - started

    grade = grade_run(agent.agent_id, run_id, eligible=case["eligible"])
    workflow_doc = grade["workflow"]
    usage = meter.usage
    return {
        "arm": arm,
        "case_id": case["case_id"],
        "eligible": case["eligible"],
        "correct": grade["correct"],
        "expected_path": grade["expected_path"],
        "actual_path": grade["actual_path"],
        "recorded_business_success": grade["recorded_business_success"],
        "outcome_agrees_with_grade": grade["outcome_agrees_with_grade"],
        "skill_activated": bool(
            workflow_doc and workflow_doc.get("skills_activated")
        ),
        "end_to_end_ms": end_to_end * 1000,
        "inference_ms": meter.inference_seconds * 1000,
        "non_inference_ms": max(
            0.0, (end_to_end - meter.inference_seconds) * 1000
        ),
        "llm_calls": len(meter.events),
        "prompt_tokens": usage.get("prompt_tokens", 0),
        "completion_tokens": usage.get("completion_tokens", 0),
        "cached_tokens": usage.get("cached_tokens", 0),
        "arm_label_visible": any(
            event["arm_label_visible"] for event in meter.events
        ),
        "raw_workflow_context_present": any(
            event["raw_workflow_context_present"] for event in meter.events
        ),
        "run_identifier_visible": any(
            event["run_identifier_visible"] for event in meter.events
        ),
        "skill_context_present": any(
            event["skill_context_present"] for event in meter.events
        ),
        "training_identifier_visible": any(
            event["training_identifier_visible"] for event in meter.events
        ),
        "response": response,
    }

In [ ]:
EVAL_CASES = [
    {
        "case_id": "refund_direct",
        "query": "Please resolve the refund for R-1011.",
        "eligible": True,
    },
    {
        "case_id": "refund_return_payment",
        "query": "Could you return the payment for order R-1012?",
        "eligible": True,
    },
    {
        "case_id": "refund_money_back",
        "query": "The customer wants their money back for R-1013.",
        "eligible": True,
    },
    {
        "case_id": "refund_completed",
        "query": "Handle the refund on completed order R-1014.",
        "eligible": True,
    },
    {
        "case_id": "refund_reverse_charge",
        "query": "Reverse the charge for R-1015 after checking the order.",
        "eligible": True,
    },
    {
        "case_id": "refund_paraphrase",
        "query": "Process R-1016's eligible return payment.",
        "eligible": True,
    },
    {
        "case_id": "status_only_explicit",
        "query": "What is the status of R-1017? Do not change anything.",
        "eligible": False,
    },
    {
        "case_id": "status_only_information",
        "query": "Check R-1018 for me; this is information only.",
        "eligible": False,
    },
    {
        "case_id": "pending_guardrail",
        "query": "Refund R-1099 only if its current state permits it.",
        "eligible": False,
    },
    {
        "case_id": "missing_guardrail",
        "query": "Refund R-1199 if it exists and is eligible.",
        "eligible": False,
    },
]

experiment_rows = []
arm_rng = random.Random(1707)
with InferenceMeter(capture_agent.model) as capture_meter, InferenceMeter(
    learning_agent.model
) as learning_meter, InferenceMeter(explicit_sop_agent.model) as explicit_sop_meter:
    for case in EVAL_CASES:
        arms = [
            ("capture_only", capture_agent, capture_meter),
            ("continual_learning", learning_agent, learning_meter),
            ("explicit_sop_control", explicit_sop_agent, explicit_sop_meter),
        ]
        arm_rng.shuffle(arms)
        for arm, agent, meter in arms:
            row = run_measured(arm, agent, meter, case)
            experiment_rows.append(row)
            print(
                arm,
                case["case_id"],
                {
                    "correct": row["correct"],
                    "path": row["actual_path"],
                    "skill_activated": row["skill_activated"],
                    "e2e_ms": round(row["end_to_end_ms"], 1),
                    "inference_ms": round(row["inference_ms"], 1),
                },
            )

outcome_disagreements = [
    (row["arm"], row["case_id"])
    for row in experiment_rows
    if not row["outcome_agrees_with_grade"]
]
if outcome_disagreements:
    raise RuntimeError(
        f"Stored business outcomes disagree with held-out grades: {outcome_disagreements}"
    )

expected_pairs = {
    (arm, case["case_id"])
    for case in EVAL_CASES
    for arm in (
        "capture_only",
        "continual_learning",
        "explicit_sop_control",
    )
}
observed_pairs = {(row["arm"], row["case_id"]) for row in experiment_rows}
if len(experiment_rows) != 30 or observed_pairs != expected_pairs:
    raise RuntimeError("Held-out result rows are incomplete or duplicated.")

context_violations = []
for row in experiment_rows:
    flags = {
        "arm_label_visible": row["arm_label_visible"],
        "run_identifier_visible": row["run_identifier_visible"],
        "raw_workflow_context_present": row["raw_workflow_context_present"],
        "training_identifier_visible": row["training_identifier_visible"],
        "skill_context_matches_attribution": (
            row["skill_context_present"] == row["skill_activated"]
        ),
    }
    if any(
        flags[name]
        for name in (
            "arm_label_visible",
            "run_identifier_visible",
            "raw_workflow_context_present",
            "training_identifier_visible",
        )
    ) or not flags["skill_context_matches_attribution"]:
        context_violations.append(
            {"arm": row["arm"], "case_id": row["case_id"], **flags}
        )
if context_violations:
    raise RuntimeError(f"Model-context audit failed: {context_violations}")

latency_violations = [
    (row["arm"], row["case_id"])
    for row in experiment_rows
    if row["inference_ms"] < 0
    or row["end_to_end_ms"] + 1e-6 < row["inference_ms"]
]
if latency_violations:
    raise RuntimeError(f"Latency accounting failed: {latency_violations}")

visible_tool_field_leaks = [
    sorted(FORBIDDEN_MODEL_FIELDS.intersection(result))
    for result in MODEL_VISIBLE_TOOL_RESULTS
    if FORBIDDEN_MODEL_FIELDS.intersection(result)
]
if visible_tool_field_leaks:
    raise RuntimeError(
        f"Internal fields appeared in model-visible tool results: {visible_tool_field_leaks}"
    )

print(
    {
        "held_out_rows": len(experiment_rows),
        "unique_arm_case_pairs": len(observed_pairs),
        "outcome_disagreements": outcome_disagreements,
        "context_violations": context_violations,
        "model_visible_tool_field_leaks": visible_tool_field_leaks,
        "primary_system_prompts_identical": primary_prompts_identical,
        "primary_tool_schemas_identical": primary_tool_schemas_identical,
    }
)

In [ ]:
def percentile(values, probability: float):
    ordered = sorted(values)
    if not ordered:
        return 0.0
    position = (len(ordered) - 1) * probability
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return ordered[lower]
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight


def wilson_interval(successes: int, total: int, z: float = 1.96):
    if total == 0:
        return (0.0, 0.0)
    proportion = successes / total
    denominator = 1 + (z * z / total)
    center = (proportion + z * z / (2 * total)) / denominator
    margin = (
        z
        * math.sqrt(
            proportion * (1 - proportion) / total
            + z * z / (4 * total * total)
        )
        / denominator
    )
    return (max(0.0, center - margin), min(1.0, center + margin))


summaries = []
for arm in ("capture_only", "continual_learning", "explicit_sop_control"):
    rows = [row for row in experiment_rows if row["arm"] == arm]
    successes = sum(row["correct"] for row in rows)
    interval = wilson_interval(successes, len(rows))
    summaries.append(
        {
            "arm": arm,
            "successes": successes,
            "cases": len(rows),
            "accuracy": successes / len(rows),
            "accuracy_ci_low": interval[0],
            "accuracy_ci_high": interval[1],
            "skill_activation_rate": sum(row["skill_activated"] for row in rows)
            / len(rows),
            "e2e_p50_ms": statistics.median(
                row["end_to_end_ms"] for row in rows
            ),
            "e2e_p95_ms": percentile(
                [row["end_to_end_ms"] for row in rows], 0.95
            ),
            "inference_p50_ms": statistics.median(
                row["inference_ms"] for row in rows
            ),
            "inference_p95_ms": percentile(
                [row["inference_ms"] for row in rows], 0.95
            ),
            "non_inference_p50_ms": statistics.median(
                row["non_inference_ms"] for row in rows
            ),
            "mean_llm_calls": statistics.mean(
                row["llm_calls"] for row in rows
            ),
            "mean_total_tokens": statistics.mean(
                row["prompt_tokens"] + row["completion_tokens"] for row in rows
            ),
        }
    )

paired_deltas = []
for case in EVAL_CASES:
    by_arm = {
        row["arm"]: row
        for row in experiment_rows
        if row["case_id"] == case["case_id"]
    }
    paired_deltas.append(
        int(by_arm["continual_learning"]["correct"])
        - int(by_arm["capture_only"]["correct"])
    )

bootstrap_rng = random.Random(1707)
bootstrap_deltas = []
for _ in range(10000):
    sample = [
        paired_deltas[bootstrap_rng.randrange(len(paired_deltas))]
        for _ in paired_deltas
    ]
    bootstrap_deltas.append(statistics.mean(sample))

paired_delta = statistics.mean(paired_deltas)
paired_ci = (
    percentile(bootstrap_deltas, 0.025),
    percentile(bootstrap_deltas, 0.975),
)
accuracy_by_arm = {row["arm"]: row["accuracy"] for row in summaries}
skill_vs_explicit_sop_delta = (
    accuracy_by_arm["continual_learning"]
    - accuracy_by_arm["explicit_sop_control"]
)

print(
    "| arm | correct | accuracy (95% Wilson CI) | activation | "
    "e2e p50 ms | e2e p95 ms | inference p50 ms | inference p95 ms | "
    "non-inference p50 ms | mean LLM calls | mean tokens |"
)
print("|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|")
for row in summaries:
    print(
        f"| {row['arm']} | {row['successes']}/{row['cases']} | "
        f"{row['accuracy']:.1%} "
        f"({row['accuracy_ci_low']:.1%}-{row['accuracy_ci_high']:.1%}) | "
        f"{row['skill_activation_rate']:.1%} | "
        f"{row['e2e_p50_ms']:.1f} | {row['e2e_p95_ms']:.1f} | "
        f"{row['inference_p50_ms']:.1f} | {row['inference_p95_ms']:.1f} | "
        f"{row['non_inference_p50_ms']:.1f} | "
        f"{row['mean_llm_calls']:.2f} | {row['mean_total_tokens']:.0f} |"
    )

print(
    {
        "paired_accuracy_delta_percentage_points": round(paired_delta * 100, 1),
        "paired_bootstrap_95_ci_percentage_points": [
            round(paired_ci[0] * 100, 1),
            round(paired_ci[1] * 100, 1),
        ],
        "skill_minus_explicit_sop_control_percentage_points": round(
            skill_vs_explicit_sop_delta * 100, 1
        ),
        "interpretation": (
            "illustrative positive result"
            if paired_delta > 0
            else "no observed accuracy benefit"
            if paired_delta == 0
            else "observed negative transfer"
        ),
    }
)

skill_retrieval_audit = []
for case in EVAL_CASES:
    matches = manager.retrieve_skills_for_query(case["query"])
    selected = next(
        (
            match
            for match in matches
            if match.skill.skill_id == promoted_skill_id
        ),
        None,
    )
    skill_retrieval_audit.append(
        {
            "case_id": case["case_id"],
            "eligible": case["eligible"],
            "selected": selected is not None,
            "similarity": round(selected.similarity, 6) if selected else None,
            "threshold": manager.config.retrieval_min_similarity,
        }
    )

case_audit_rows = [
    {
        key: row[key]
        for key in (
            "arm",
            "case_id",
            "eligible",
            "correct",
            "expected_path",
            "actual_path",
            "recorded_business_success",
            "outcome_agrees_with_grade",
            "skill_activated",
            "skill_context_present",
            "raw_workflow_context_present",
            "arm_label_visible",
            "run_identifier_visible",
            "training_identifier_visible",
            "end_to_end_ms",
            "inference_ms",
            "non_inference_ms",
            "llm_calls",
            "prompt_tokens",
            "completion_tokens",
            "cached_tokens",
        )
    }
    for row in experiment_rows
]

print("CASE_AUDIT_JSON_BEGIN")
print(json.dumps(case_audit_rows, indent=2, sort_keys=True))
print("CASE_AUDIT_JSON_END")
print("SKILL_RETRIEVAL_AUDIT_JSON_BEGIN")
print(json.dumps(skill_retrieval_audit, indent=2, sort_keys=True))
print("SKILL_RETRIEVAL_AUDIT_JSON_END")

#### Execution Status

This notebook now installs the newest published MemoRizz distribution before importing
the package. Outputs from the earlier execution were cleared when that installation path
changed. The notebook has **not** been rerun, so it intentionally reports no current
accuracy or latency result.

Run every cell in order to produce a new installed-package result. The install and import
cells should record the resolved package version and a location under `site-packages` or
`dist-packages`. The evaluation cells then print the arm summary, paired bootstrap
interval, exact 30-case audit JSON, and skill-retrieval audit.

Before treating a new output as evidence, confirm all of the following:

- all 19 code cells completed once, in order, without an error;
- all Oracle vector columns match the configured in-database embedding dimension;
- both primary system prompts and tool schemas are identical before treatment context;
- the model-context audit reports no arm label, run ID, training ID, or raw workflow leak;
- stored business outcomes agree with the independent grader for all 30 held-out runs;
- end-to-end and provider-inference latency are reported separately; and
- teardown runs even when an earlier experimental cell fails.

The prior validation work remains encoded as runtime assertions rather than copied result
text. Report only the figures produced by the installed version shown in the same run.

#### How to Read the Next Results

The causal claim is intentionally narrow. The two primary arms receive the same real,
coached demonstrations and the same production prompt; the treatment arm additionally
receives the reviewed skill. A positive paired result would therefore support the proposition
that compiling these passive workflows into a retrieved instruction improved this SOP
benchmark. The explicit-SOP control shows how much of that gap could be closed simply by
putting the complete policy in the prompt.

##### Threats to Validity

This test deliberately creates learning headroom: the receipt requirement is present in
the coached seed instruction and absent from the later production instruction. That is a
valid test of retaining and compiling a taught SOP, but it is not a neutral test of
general agent improvement. The benchmark was also iterated after an earlier negative
three-case run; it was not preregistered. The seed and evaluation queries use the same
domain and tool family, so this is within-distribution transfer to new IDs and
paraphrases, not cross-domain learning.

Promotion is conditional on passing evidence gates, model validation, manual shadow
review, and canaries. That is appropriate product behavior, but any reported result is
conditional on a skill surviving those gates. The case-resampled bootstrap
interval is descriptive for these ten authored cases, not confirmatory statistical
evidence. No grader label, private arm/run identifier, internal business-event log, or
score is fed to the LLM during a held-out run. The LLM sees only the normal business
fields returned by each tool. The same outcome callback is applied to every arm and
checked against the separate grader after each run.

A completed controlled contrast and its case-level paths can provide **evidence of an
improvement for the tested agent and benchmark**. An accuracy increase alone would not
prove causation; attribution also depends on equal primary prompts and tools, absence of context
leakage, paired cases, and the independent business grader. It does **not** establish a
universal improvement across agents, models, or domains. Inspect:

- case-level paths and business events, especially pending and missing-order controls;
- skill activation coverage, because an unactivated skill cannot cause a treatment
  effect;
- the paired accuracy delta and its wide small-sample interval;
- provider inference and end-to-end latency separately; and
- whether added prompt context increased tokens or latency despite better accuracy.

For a stronger claim, freeze the prompt, cases, rubric, promotion settings, and analysis
before execution; repeat randomized trials across models and domains; and evaluate on
production outcomes. If the skill hurts guardrails or fails canaries, demote it rather
than changing the rubric.

### Part V Key Takeaways

- The paired design isolates learned-skill availability from passive workflow capture.
- The explicit-SOP arm shows whether a learned skill recovers the policy available in writing.
- Report accuracy, activation, tokens, end-to-end latency, and inference latency from the same fresh run.

**Continue to Part VI:** after a fresh execution, preserve its notebook evidence while
removing only run-owned database records so the next run starts from a known state.

---

## Part VI - Clean Up and Reproduce

Part V defines how to measure change and bound the claim. The last part closes the operational
loop by deleting benchmark data without damaging shared Oracle infrastructure.

### 9. Teardown

This teardown deletes Toolbox, Skillbox, workflow, conversation, and tool-log rows owned
by the three unique agent IDs created for this run. It does not call `delete_all`, drop
MemoRizz tables, or remove `ALL_MINILM_L12_V2`, because those resources may be shared.

If an earlier cell fails, fix the root cause and run this cell manually before restarting
the experiment.

In [ ]:
DEMO_AGENT_IDS = {LEARNING_AGENT_ID, CAPTURE_AGENT_ID, EXPLICIT_SOP_AGENT_ID}
ID_FIELDS = {
    MemoryType.TOOLBOX: "tool_id",
    MemoryType.SKILLBOX: "skill_id",
    MemoryType.WORKFLOW_MEMORY: "_id",
    MemoryType.CONVERSATION_MEMORY: "_id",
    MemoryType.TOOL_LOG: "_id",
}
deleted = defaultdict(int)

for memory_type, id_field in ID_FIELDS.items():
    rows = provider.list_all(memory_store_type=memory_type) or []
    for row in rows:
        if row.get("agent_id") not in DEMO_AGENT_IDS:
            continue
        record_id = row.get(id_field)
        if record_id and provider.delete_by_id(
            str(record_id),
            memory_store_type=memory_type,
        ):
            deleted[memory_type.value] += 1

for agent_id in DEMO_AGENT_IDS:
    if provider.delete_memagent(agent_id, cascade=False):
        deleted[MemoryType.MEMAGENT.value] += 1

LOOKED_UP.clear()
REFUNDED.clear()
REFUND_EVENTS.clear()
RECEIPT_EVENTS.clear()
MODEL_VISIBLE_TOOL_RESULTS.clear()
provider.close()

print("Deleted guide rows:", dict(deleted))
print("Retained shared Oracle schema and ONNX embedding model.")

### Part VI Key Takeaways

- Teardown removes only rows owned by the three benchmark agents.
- The Oracle schema and ONNX model remain available for later runs.
- Reproduction requires a fresh run ID, verified credentials, and a verified installed-package import.

**Continue the learning loop:** a future execution begins again with new workflow evidence,
then repeats the same capture, distillation, governance, evaluation, and cleanup discipline.